In [1]:
from sklearn import logger

from pyspark.sql import SparkSession
from utils.dataPipeline import DataPreparation
from utils.ALSModelPipeline import ALSModelPipeline

spark = SparkSession.builder \
    .appName("popularity_exploration") \
    .config("spark.sql.shuffle.partitions", "4000") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .config("spark.executor.failuresValidityInterval", "1h")\
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/04/22 05:24:18 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/22 05:24:18 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 05:24:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [2]:
config = {
    "base_date_str": "2026-03-28",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als",
    "test_days": 30,
    "train_days": 90,
    "playtime_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "click_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new",
    "distinct_user_content_threshold": 1
}

data_prep = DataPreparation(spark, config, True)
data_prep.load_data_checkpoint()
# 2. Load Model Checkpoint (Restores pre-computed als_top_k)
als_pipeline = ALSModelPipeline(spark, config, data_prep=data_prep)
als_pipeline.load_model_checkpoint()

# 3. Serve Recs Instantly
als_pipeline.get_recommendations_userId("ib-xPv06AjuYOE4BI0")

Loading data checkpoint from: gs://wynk-ml-workspace/_temp/harshith/als/default_checkpoint...


Checkpoint loaded successfully! Ready for training or inference.
Loading ALS predictions checkpoint from: gs://wynk-ml-workspace/_temp/harshith/als/default_checkpoint/als_top_k...
Model checkpoint loaded successfully! Ready for inference.

Watch/Click History for userId ib-xPv06AjuYOE4BI0:


26/04/20 10:42:30 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Watch History Count:  2


26/04/20 10:42:39 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------------------------------------------------------+------------------------+-------------------------------------+
|title                                                 |item_id                 |Genres                               |
+------------------------------------------------------+------------------------+-------------------------------------+
|Coco                                                  |HOTSTAR_DTH_MOVIE_485462|[Family, Animation, Music, Adventure]|
|Pirates of the Caribbean: The Curse of the Black Pearl|HOTSTAR_DTH_MOVIE_3426  |[Adventure, Fantasy, Action]         |
+------------------------------------------------------+------------------------+-------------------------------------+



User ID ib-xPv06AjuYOE4BI0 not found in user lookup.


[]

## Todo:
1. recommend for all users and check the content coverage, meaning out of all the contents, check how many are actually covered. Check only if the content is covered or not
2. Out of the recommendatoins, for each content, get the count of how many are being recommended. then get the distribution. idk what to do with this distribution. 

### Load ALS Artifacts

In [5]:
config = {
    "base_date_str": "2026-03-28",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als",
    "custom_path":"gs://wynk-ml-workspace/_temp/harshith/als_custom",
    "test_days": 30,
    "train_days": 90,
    "playtime_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "click_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new",
    "distinct_user_content_threshold": 1
}

data_prep = DataPreparation(spark, config, True)
data_prep.load_data_checkpoint()

als_pipeline = ALSModelPipeline(spark, config, data_prep=data_prep)
als_model, user_lookup, item_lookup, als_top_50 = als_pipeline.load_als_artifacts(base_save_path="gs://wynk-ml-workspace/_temp/harshith/als",als_param_k=50)

als_top_50.cache()
als_top_50.count()

Loading data checkpoint from: gs://wynk-ml-workspace/_temp/harshith/als/default_checkpoint...


26/04/20 10:58:12 WARN CacheManager: Asked to cache already cached data.        
26/04/20 10:58:13 WARN CacheManager: Asked to cache already cached data.


Checkpoint loaded successfully! Ready for training or inference.
Loading model and lookups from gs://wynk-ml-workspace/_temp/harshith/als...


26/04/20 10:58:15 WARN CacheManager: Asked to cache already cached data.
26/04/20 10:58:15 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Successfully loaded all artifacts!


26/04/20 11:01:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 6 for reason Executor for container container_1764236692086_7388_01_000007 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/20 11:01:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 7 for reason Executor for container container_1764236692086_7388_01_000008 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/20 11:04:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 9 for reason Executor for container container_1764236692086_7388_01_000010 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/20 11:04:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 8 for reason Executor for container container_1764236692086_7388_

2531783

In [6]:
custom_model_path = config['custom_path']
checkpoint_path = f"{custom_model_path}/als_top_k=50"

als_top_50.write.mode("overwrite").parquet(checkpoint_path)

In [4]:
als_top_50.count()

2531783

In [5]:
user_lookup.count()

2531783

In [7]:
item_lookup = data_prep.item_lookup
item_lookup.show()

+--------------------+---------+
|             item_id|itemIndex|
+--------------------+---------+
|AHA_MOVIE_6A258B1...|    40161|
|ULTRA_MOVIE_689b1...|    40162|
|HOTSTAR_DTH_MOVIE...|    40163|
|HOTSTAR_DTH_TVSHO...|    40164|
|CHAUPAL_MOVIE_en_...|    40165|
|ZEEFIVE_MOVIE_0-0...|    40166|
|HUNGAMA_TVSHOW_57...|    40167|
|ZEEFIVE_TVSHOW_0-...|    40168|
|MANORAMAMAX_MOVIE...|    40169|
|  SUNNXT_MOVIE_12943|    40170|
|ZEEFIVE_MOVIE_0-0...|    40171|
|ZEEFIVE_MOVIE_0-0...|    40172|
|  DOCUBAY_MOVIE_4164|    40173|
|ZEEFIVE_MOVIE_0-0...|    40174|
|ZEEFIVE_MOVIE_0-0...|    40175|
|LIONSGATEPLAY_MOV...|    40176|
|HOTSTAR_DTH_TVSHO...|    40177|
|MANORAMAMAX_MOVIE...|    40178|
|  SUNNXT_MOVIE_12953|    40179|
|  EPICON_MOVIE_26001|    40180|
+--------------------+---------+
only showing top 20 rows



In [ ]:
from pyspark.sql import functions as F

# Assuming your DataFrames are named:
# 1. recs_df (columns: "userId", "recommended_items" which is an array)
# 2. items_df (columns: "item_id", "title", etc.)

# Step 1: Explode the array so each recommended item gets its own row
exploded_recs = als_top_50.select(
    "userIndex", 
    F.explode("predicted_items").alias("itemIndex")
)

# Step 2: Group by the item and count how many times it appears
item_counts = exploded_recs.groupBy("itemIndex") \
    .agg(F.count("userIndex").alias("times_recommended"))

# Step 3: Join back to your main items DataFrame to attach metadata (like titles)
# We use a left join so you keep all items, even those recommended 0 times
final_counts_df = item_lookup.join(item_counts, on="itemIndex", how="left") \
    .fillna({"times_recommended": 0}) \
    .orderBy(F.col("times_recommended").desc())

# View the results
final_counts_df.show(50,truncate=False)

26/04/20 11:17:38 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+---------+---------------------------------------------------------------+-----------------+
|itemIndex|item_id                                                        |times_recommended|
+---------+---------------------------------------------------------------+-----------------+
|29321    |PLAYFLIX_MOVIE_ATL0000000643                                   |790397           |
|37027    |ZEEFIVE_MOVIE_0-0-136809                                       |782640           |
|25280    |MINITV_MOVIE_amzn1.dv.gti.208513a2-8513-4d4f-8949-904f60b331fe |773821           |
|28715    |HOTSTAR_DTH_MOVIE_985031                                       |757599           |
|29755    |KLIKK_MOVIE_2508                                               |718111           |
|33202    |SONYLIV_VOD_SPORTS_CLIPS_1090497910                            |662033           |
|14284    |AHA_MOVIE_C1E412F9-4EE5-4D7E-AFFB-EFBC04C63B33                 |658574           |
|39750    |HOTSTAR_DTH_TVSHOW_3551                          

In [10]:
coverage = final_counts_df.filter(F.col("times_recommended")>0).count()/final_counts_df.count()

print("coverage: ", coverage*100)

26/04/20 11:21:28 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/20 11:21:29 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


coverage:  5.753742773852463


In [17]:
final_counts_df.filter(F.col("times_recommended")>0).orderBy("times_recommended").show(600)

26/04/20 11:27:10 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+---------+--------------------+-----------------+
|itemIndex|             item_id|times_recommended|
+---------+--------------------+-----------------+
|     2941|SHEMAROOME_MOVIE_...|                1|
|    25337|  SUNNXT_MOVIE_12625|                1|
|    13649|   SUNNXT_MOVIE_7855|                1|
|    24236|EROSNOW_MOVIE_671...|                1|
|    21924|HOTSTAR_DTH_MOVIE...|                1|
|    18797|EROSNOW_MOVIE_685...|                1|
|    21947|HOTSTAR_DTH_TVSHO...|                1|
|    21988|   SUNNXT_MOVIE_7264|                1|
|    10465|SONYLIV_VOD_SPORT...|                1|
|    24698|   SUNNXT_MOVIE_8774|                1|
|     5886|    KLIKK_MOVIE_1790|                1|
|     2431|EROSNOW_MOVIE_694...|                1|
|       40|   SUNNXT_MOVIE_9914|                1|
|    25895|CHAUPAL_MOVIE_en_...|                1|
|    22059|HOTSTAR_DTH_TVSHO...|                1|
|    23125|SONYLIV_VOD_SPORT...|                1|
|    26543|MINITV_MOVIE_amzn...

In [11]:
item_lookup.count()

40478

In [18]:
# Define the percentiles you want (expressed as decimals 0.0 to 1.0)
percentiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

# Calculate the values
# 0.01 is the relative error: lower is more accurate but slower
quantile_values = final_counts_df.filter(F.col("times_recommended")>0).stat.approxQuantile(
    "times_recommended", 
    percentiles, 
    0.01
)

# Pair them up and print the results
print("--- User Content Consumption Percentiles ---")
for p, v in zip(percentiles, quantile_values):
    print(f"{int(p*100)}th Percentile: {int(v)} contents")

26/04/20 11:27:59 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


--- User Content Consumption Percentiles ---
25th Percentile: 2327 contents
50th Percentile: 16270 contents
75th Percentile: 55533 contents
90th Percentile: 133491 contents
95th Percentile: 219108 contents
99th Percentile: 790397 contents


Checking total watchtime distribution for users:

In [2]:
config = {
    "base_date_str": "2026-03-28",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als",
    "custom_path":"gs://wynk-ml-workspace/_temp/harshith/als_custom",
    "test_days": 30,
    "train_days": 90,
    "playtime_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "click_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new",
    "distinct_user_content_threshold": 1
}

data_prep = DataPreparation(spark, config)
data_prep.prepare_features_and_metadata()

Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-06
Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-05


Data read from gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new completed, filtering for common users...
self train and test df have been assigned
Playtime combined successfully.
Playtime combined successfully.
Playtime to label completed.
Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new/day=2026-02-06
Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new/day=2026-02-05
Data read from gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new completed, filtering for common users...
Successfully combined training and testing features!


26/04/22 05:25:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [6]:
data_prep.load_data_checkpoint()

Loading data checkpoint from: gs://wynk-ml-workspace/_temp/harshith/als/default_checkpoint...
Checkpoint loaded successfully! Ready for training or inference.


In [5]:
data_prep.combined_train_df.show(5)

+------------------+--------------------+----------+
|            userId|             item_id|confidence|
+------------------+--------------------+----------+
|cHQyYOv7W0b_BGGmr0|AHA_MOVIE_A2D5FA0...|       0.2|
|oTeQG6kK2VP1S9SSJ0|AHA_MOVIE_BD3B8A3...|       0.2|
|Kh-v2AYcquep68xR60|CHAUPAL_MOVIE_en_...|       2.0|
|6tOXwCKirrw5wOBHF0|CHAUPAL_MOVIE_en_...|       4.0|
|GMUsr3f1qXBKU7fCc0|EROSNOW_MOVIE_676...|       0.2|
+------------------+--------------------+----------+
only showing top 5 rows



In [15]:
from pyspark.sql.functions import col, sum
from pyspark.sql import functions as F

In [13]:
user_total_watchtime = data_prep.train_df.groupBy("userId").agg(sum(col("total_play_time_sec")).alias("total_user_watchtime"))
user_total_watchtime.show(5)

+------------------+--------------------+
|            userId|total_user_watchtime|
+------------------+--------------------+
|--JPsA7HSE0uH3hy10|           27421.782|
|--ePPmrr8SAYWtmaJ0|  55992.024999999994|
|--flnr9BU4PlIxi1s0|             228.909|
|--gqgOw07pjvLZl_C0|           98529.884|
|--jv9olwuRzSmj8mq0|             33032.0|
+------------------+--------------------+
only showing top 5 rows



In [18]:
# Define the percentiles you want (expressed as decimals 0.0 to 1.0)
percentiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

# Calculate the values
# 0.01 is the relative error: lower is more accurate but slower
quantile_values = user_total_watchtime.filter(F.col("total_user_watchtime")>0).stat.approxQuantile(
    "total_user_watchtime", 
    percentiles, 
    0.01
)

# Pair them up and print the results
print("--- User Content Consumption Percentiles ---")
for p, v in zip(percentiles, quantile_values):
    print(f"{int(p*100)}th Percentile: {v} seconds")

--- User Content Consumption Percentiles ---
25th Percentile: 9686.738 seconds
50th Percentile: 33928.225999999995 seconds
75th Percentile: 85287.10599999999 seconds
90th Percentile: 164810.75199999998 seconds
95th Percentile: 234953.238 seconds
99th Percentile: 3030204031671.885 seconds


In [20]:
user_total_watchtime.filter(col("total_user_watchtime") == 3030204031671.885).show()

+------------------+--------------------+
|            userId|total_user_watchtime|
+------------------+--------------------+
|25nKgd9wUM19H4OEl0|3.030204031671885E12|
+------------------+--------------------+



In [ ]:
data_prep.train_df.filter(col("userId") == "25nKgd9wUM19H4OEl0").show(100, truncate = False)

26/04/22 05:55:09 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------------------+-----------------------------+-------------------+----------+
|userId            |item_id                      |total_play_time_sec|day       |
+------------------+-----------------------------+-------------------+----------+
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|1005.69            |2026-02-01|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|3714.1179999999995 |2026-02-02|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|3007.571           |2026-02-14|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|8162.203           |2026-02-21|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|2933.5150000000003 |2026-02-25|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|6110.033           |2026-02-10|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_MOVIE_1090503578 |5594.696           |2026-02-22|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700001090|2860.618           |2026-02-12|
|25nKgd9wUM19H4OEl0|SONYLIV_VOD_TVSHOW_1700000725|4032.6690000000003 |2026-02-13|
|25nKgd9wUM19H4O

26/04/22 05:56:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 93 for reason Executor for container container_1764236692086_7475_01_000101 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 05:56:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 92 for reason Executor for container container_1764236692086_7475_01_000100 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 05:56:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 88 for reason Executor for container container_1764236692086_7475_01_000095 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 05:56:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 83 for reason Executor for container container_1764236692086_7